<a href="https://colab.research.google.com/github/Bijon2002/Airline-Tweet-Sentiment-Analyzer/blob/main/sentiment_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import nltk
import string
import re
import pickle

from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

print("✅ Libraries imported!")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


✅ Libraries imported!


In [3]:
df = pd.read_csv('/content/drive/MyDrive/Tweets.csv')

# Keep only what we need
df = df[["airline_sentiment", "text"]]

print(df.shape)
print(df.head())
print(df['airline_sentiment'].value_counts())

(14640, 2)
  airline_sentiment                                               text
0           neutral                @VirginAmerica What @dhepburn said.
1          positive  @VirginAmerica plus you've added commercials t...
2           neutral  @VirginAmerica I didn't today... Must mean I n...
3          negative  @VirginAmerica it's really aggressive to blast...
4          negative  @VirginAmerica and it's a really big bad thing...
airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64


In [4]:
ps = PorterStemmer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http.?://[^\s]+[\s]?', '', text)
    text = nltk.word_tokenize(text)
    y = []
    for i in text:
        if i not in stopwords.words('english'):
            y.append(i)
    text = y[:]
    y.clear()
    for i in text:
        y.append(ps.stem(i))
    return " ".join(y)

print("✅ clean_text function ready!")

✅ clean_text function ready!


In [5]:
df['text_cleaned'] = df['text'].apply(clean_text)

print(df[['text', 'text_cleaned']].head())

                                                text  \
0                @VirginAmerica What @dhepburn said.   
1  @VirginAmerica plus you've added commercials t...   
2  @VirginAmerica I didn't today... Must mean I n...   
3  @VirginAmerica it's really aggressive to blast...   
4  @VirginAmerica and it's a really big bad thing...   

                                        text_cleaned  
0                  @ virginamerica @ dhepburn said .  
1  @ virginamerica plu 've ad commerci experi ......  
2  @ virginamerica n't today ... must mean need t...  
3  @ virginamerica 's realli aggress blast obnoxi...  
4            @ virginamerica 's realli big bad thing  


In [6]:
tfidf = TfidfVectorizer(max_features=3000)

X = tfidf.fit_transform(df['text_cleaned']).toarray()
Y = df['airline_sentiment'].values

print(f"X shape: {X.shape}")
print(f"Y shape: {Y.shape}")

X shape: (14640, 3000)
Y shape: (14640,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, Y,
    test_size=0.2,
    random_state=2
)

print(f"Train size: {X_train.shape}")
print(f"Test size: {X_test.shape}")

Train size: (11712, 3000)
Test size: (2928, 3000)


In [8]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)
nb_accuracy = accuracy_score(y_test, y_pred_nb)

print(f"✅ Naive Bayes Accuracy: {nb_accuracy * 100:.2f}%")

✅ Naive Bayes Accuracy: 72.13%


In [9]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred)
print(f"✅ Random Forest Accuracy: {rf_accuracy * 100:.2f}%")

✅ Random Forest Accuracy: 75.00%


In [10]:
with open('sentiment_model.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)

print("✅ Model saved!")
print("✅ Vectorizer saved!")

✅ Model saved!
✅ Vectorizer saved!


In [11]:
from sklearn.ensemble import RandomForestClassifier

model_v2 = RandomForestClassifier(
    n_estimators=200,      # more trees = smarter
    max_depth=20,          # deeper thinking
    random_state=2
)
model_v2.fit(X_train, y_train)
y_pred_v2 = model_v2.predict(X_test)
print(f"✅ RF v2 Accuracy: {accuracy_score(y_test, y_pred_v2) * 100:.2f}%")

✅ RF v2 Accuracy: 65.20%


In [12]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(max_iter=1000, random_state=2)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)
print(f"✅ Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr) * 100:.2f}%")

✅ Logistic Regression Accuracy: 77.36%


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

# More features
tfidf_v2 = TfidfVectorizer(max_features=8000, ngram_range=(1,2))
X_v2 = tfidf_v2.fit_transform(df['text_cleaned']).toarray()
Y = df['airline_sentiment'].values

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_v2, Y, test_size=0.2, random_state=2
)

lr_v2 = LogisticRegression(
    max_iter=1000,
    C=5,                    # stronger learning
    class_weight='balanced', # fix imbalance
    random_state=2
)
lr_v2.fit(X_train2, y_train2)
y_pred_v2 = lr_v2.predict(X_test2)
print(f"✅ LR v2 Accuracy: {accuracy_score(y_test2, y_pred_v2) * 100:.2f}%")

✅ LR v2 Accuracy: 75.31%


In [15]:
neutral_tweets_df = df[df['airline_sentiment'] == 'neutral']
unique_neutral_tweets = neutral_tweets_df['text'].nunique()
print(f"Number of unique tweets with 'neutral' sentiment: {unique_neutral_tweets}")

Number of unique tweets with 'neutral' sentiment: 3067


In [16]:
print(df['airline_sentiment'].value_counts())
print("\nUnique neutral tweets:")
print(df[df['airline_sentiment'] == 'neutral']['text'].nunique())

airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64

Unique neutral tweets:
3067


In [13]:
# In Cell 5, change max_features from 3000 to 5000
tfidf = TfidfVectorizer(max_features=5000)